# Python Drills, Part 2: NumPy and Gaussian Elimination

Part 1 did vectors and matrices with plain lists and loops. This notebook does the same kind of work
with NumPy, which is what every ML library is built on, and then finishes with a from-scratch linear
system solver.

The format is the same as Part 1: read and run the examples, fill in each `# YOUR CODE HERE`, run the
test cell under it, and do not edit the tests. Some exercises ask you to use loops on purpose and
others ask you to avoid loops entirely; each one says which.

Run the setup cell first.

In [ ]:
# Setup: run this cell first. You do not need to read or change it.
import math, numbers
import numpy as np

_results = {}

def _same(got, expected, tol):
    if isinstance(got, np.ndarray) or isinstance(expected, np.ndarray):
        try:
            g = np.asarray(got, dtype=float)
            e = np.asarray(expected, dtype=float)
        except (TypeError, ValueError):
            return False
        return g.shape == e.shape and np.allclose(g, e, rtol=tol, atol=tol)
    if isinstance(expected, (list, tuple)) and isinstance(got, (list, tuple)):
        return len(got) == len(expected) and all(_same(g, e, tol) for g, e in zip(got, expected))
    if isinstance(expected, dict) and isinstance(got, dict):
        return got.keys() == expected.keys() and all(_same(got[k], expected[k], tol) for k in expected)
    if (isinstance(expected, numbers.Number) and isinstance(got, numbers.Number)
            and not isinstance(expected, (bool, np.bool_)) and not isinstance(got, (bool, np.bool_))):
        return math.isclose(got, expected, rel_tol=tol, abs_tol=tol)
    return bool(got == expected)

def check(name, got, expected, tol=1e-6):
    """Compare got to expected and print PASS or FAIL. If got is a function, it is called first.
    Arrays and lists are compared by shape and values (with a small tolerance)."""
    if callable(got):
        try:
            got = got()
        except Exception as e:
            _results[name] = False
            print(f"FAIL  {name}\n        raised {type(e).__name__}: {e}")
            return
    ok = _same(got, expected, tol)
    _results[name] = ok
    if ok:
        print(f"PASS  {name}")
    else:
        print(f"FAIL  {name}\n        expected: {expected!r}\n        got:      {got!r}")
        if got is None:
            print("        (got None: is the function still a stub, or did you forget to return?)")

def check_raises(name, func, exc_type):
    """Pass if calling func() raises exc_type."""
    try:
        func()
    except exc_type:
        ok, msg = True, ""
    except Exception as e:
        ok, msg = False, f"raised {type(e).__name__} instead of {exc_type.__name__}"
    else:
        ok, msg = False, f"did not raise {exc_type.__name__}"
    _results[name] = ok
    print(f"PASS  {name}" if ok else f"FAIL  {name}\n        {msg}")

def report():
    passed = sum(_results.values())
    print(f"{passed} / {len(_results)} tests passed")
    for name, ok in _results.items():
        if not ok:
            print("  failing:", name)

print("Setup done. NumPy version", np.__version__)

## Section 6: NumPy

### 6a. Arrays are not lists

A NumPy array looks like a list, but arithmetic on it works element by element, the way it does on
vectors in math. The same operators mean something completely different on lists.

In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]
print("lists:  a + b =", a + b, "   a * 2 =", a * 2)

x = np.array(a)
y = np.array(b)
print("arrays: x + y =", x + y, "   x * 2 =", x * 2)
print("x * y =", x * y, "   x ** 2 =", x ** 2, "   y / x =", y / x)

Every array has a `shape` (a tuple of sizes, one per dimension), an `ndim` (the number of dimensions),
and a `dtype` (the type of every element). A 2D array is built from a list of rows.

Watch the `dtype`. An array built from integers stays an integer array, so storing a float into it
silently chops off the decimal part. Use `dtype=float` (or write `1.0` instead of `1`) when you need
floats.

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M)
print("shape", M.shape, " ndim", M.ndim, " dtype", M.dtype)

v = np.array([1, 2, 3])
v[0] = 2.7
print(v, "<- the 2.7 became 2")

w = np.array([1, 2, 3], dtype=float)
w[0] = 2.7
print(w)

Common ways to create arrays:

In [ ]:
print(np.zeros(4))
print(np.zeros((2, 3)))       # note the shape is passed as a tuple
print(np.ones((2, 2)))
print(np.arange(0, 10, 2))    # like range, but gives an array
print(np.linspace(0, 1, 5))   # 5 evenly spaced points from 0 to 1, both ends included
print(np.eye(3))              # identity matrix
print(np.arange(6).reshape(2, 3))
print(np.arange(6).reshape(3, -1))   # -1 means "figure out this size for me"

### Exercise 6.1: Making arrays
Replace each `None` with a single NumPy expression.

In [ ]:
a1 = None   # the integers 0 through 9
a2 = None   # 5 evenly spaced values from 0 to 1 (0, 0.25, ..., 1)
a3 = None   # a 3 by 4 array of ones
a4 = None   # the integers 0 through 11 arranged as 3 rows of 4
a5 = None   # a 2 by 5 array of integer zeros (dtype int)

In [ ]:
check("6.1 a1", a1, np.arange(10))
check("6.1 a2", a2, [0, 0.25, 0.5, 0.75, 1])
check("6.1 a3 shape", lambda: a3.shape, (3, 4))
check("6.1 a3 values", a3, np.ones((3, 4)))
check("6.1 a4", a4, [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]])
check("6.1 a5 shape", lambda: a5.shape, (2, 5))
check("6.1 a5 is an integer array", lambda: np.issubdtype(a5.dtype, np.integer), True)

### 6b. Elementwise operations and broadcasting

Math functions like `np.sqrt`, `np.exp`, and `np.abs` apply to every element. Comparisons also work
element by element and give an array of `True`/`False`.

In [ ]:
x = np.array([1.0, 4.0, 9.0, 16.0])
print(np.sqrt(x))
print(np.exp(np.array([0, 1])))
print(x > 5)
print(3 * x ** 2 - 1)    # the whole expression is computed elementwise, no loop

**Broadcasting** is how NumPy combines arrays of different shapes. The simplest case is an array and a
single number, but it also works for a matrix and a row: the row is reused for every row of the
matrix. The rule is that NumPy lines up the shapes from the right, and each pair of sizes must either
match or be 1. A size-1 dimension gets stretched to fit.

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
row = np.array([10, 20, 30])        # shape (3,)
print(M + row)                      # (2, 3) + (3,): row added to each row

col = np.array([[100], [200]])      # shape (2, 1)
print(M + col)                      # (2, 3) + (2, 1): column added to each column

r = np.array([1, 2, 3])
print(r[:, None].shape, r[None, :].shape)   # None adds a size-1 dimension
print(r[:, None] * r[None, :])               # (3, 1) * (1, 3) -> (3, 3) multiplication table

### Exercise 6.2: `poly`
Return $3x^2 - 2x + 1$ for an array `x`, computed for every element at once. No loops. It should also
work if `x` is a plain number.

In [ ]:
def poly(x):
    """Return 3x^2 - 2x + 1, elementwise."""
    # YOUR CODE HERE
    pass

In [ ]:
check("6.2 poly array", poly(np.array([0, 1, 2])), [1, 2, 9])
check("6.2 poly number", poly(3), 22)
check("6.2 poly 2D", poly(np.array([[0.5, -1.0]])), [[0.75, 6.0]])

### Exercise 6.3: `normalize`
Return the unit vector in the direction of `v`, that is $v / \lVert v \rVert$ where
$\lVert v \rVert = \sqrt{\sum_i v_i^2}$. Compute the length yourself with `np.sqrt` and `np.sum`
(do not use `np.linalg.norm` this time). No loops. You can assume `v` is not all zeros.

In [ ]:
def normalize(v):
    """Return v divided by its length."""
    # YOUR CODE HERE
    pass

In [ ]:
check("6.3 normalize([3, 4])", normalize(np.array([3, 4])), [0.6, 0.8])
check("6.3 normalize([0, 0, 5])", normalize(np.array([0, 0, 5])), [0, 0, 1])
check("6.3 result has length 1", lambda: np.sum(normalize(np.array([1.0, 2.0, 3.0])) ** 2), 1.0)
check("6.3 works on a list", lambda: normalize([1, 1]), [1 / math.sqrt(2), 1 / math.sqrt(2)])

### 6c. Indexing, masks, and reductions

2D arrays are indexed with `M[row, col]`. A colon means "everything along this dimension," so
`M[:, 1]` is column 1. Slices work in each dimension.

In [ ]:
M = np.arange(12).reshape(3, 4)
print(M)
print(M[1, 2])       # one element
print(M[0])          # row 0
print(M[:, 1])       # column 1
print(M[:2, 1:3])    # rows 0-1, columns 1-2
print(M[-1, :])      # last row

A **boolean mask** is an array of `True`/`False` used as an index. It picks out the elements where the
mask is `True`. This replaces a lot of loop-and-if code.

In [ ]:
x = np.array([3, -1, 4, -1, 5, -9, 2])
mask = x > 0
print(mask)
print(x[mask])          # just the positives
print(x[x < 0])         # usually written in one step
print(np.sum(x > 0))    # True counts as 1, so this counts the positives
y = x.copy()
y[y < 0] = 0            # set the negatives to zero
print(y)

**Reductions** like `sum`, `mean`, `max`, `min`, `std`, and `argmax` collapse an array to a number.
With `axis=0` they collapse the rows and give one answer per column; with `axis=1` they collapse the
columns and give one answer per row. In ML, rows are usually samples and columns are features, so
`axis=0` means "per feature."

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.sum())            # everything
print(M.sum(axis=0))      # column sums -> shape (3,)
print(M.sum(axis=1))      # row sums -> shape (2,)
print(M.mean(axis=0), M.max(axis=1), M.argmax(axis=1))

### Exercise 6.4: `column_means` and `row_maxes`
Two one-liners. No loops.

In [ ]:
def column_means(X):
    """Return the mean of each column of X."""
    # YOUR CODE HERE
    pass

def row_maxes(X):
    """Return the largest value in each row of X."""
    # YOUR CODE HERE
    pass

In [ ]:
X = np.array([[1, 10, 4],
              [3, 20, 2]])
check("6.4 column_means", column_means(X), [2, 15, 3])
check("6.4 row_maxes", row_maxes(X), [10, 20])
check("6.4 row_maxes one column", row_maxes(np.array([[5], [-1]])), [5, -1])

### Exercise 6.5: `count_and_mean_above`
Return a tuple `(count, mean)`: how many elements of `x` are strictly greater than `t`, and the mean of
just those elements. If no element is above `t`, return `(0, 0.0)`. Use a mask, not a loop.

In [ ]:
def count_and_mean_above(x, t):
    """Return (number of elements > t, mean of those elements)."""
    # YOUR CODE HERE
    pass

In [ ]:
check("6.5 count_and_mean_above", count_and_mean_above(np.array([1, 5, 2, 8, 3]), 2.5), (3, 16 / 3))
check("6.5 none above", count_and_mean_above(np.array([1, 2]), 10), (0, 0.0))
check("6.5 strictly greater", count_and_mean_above(np.array([4, 4, 6]), 4), (1, 6.0))

### Exercise 6.6: `standardize_columns`
Standardizing a feature means subtracting its mean and dividing by its standard deviation, so it ends
up with mean 0 and standard deviation 1. Do this for every column of `X` at once, using `axis=0` and
broadcasting. Use `np.std` with its default settings. No loops. You can assume no column is constant.

In [ ]:
def standardize_columns(X):
    """Return X with each column shifted to mean 0 and scaled to standard deviation 1."""
    # YOUR CODE HERE
    pass

In [ ]:
X = np.array([[1.0, 10.0],
              [2.0, 20.0],
              [3.0, 30.0]])
z = math.sqrt(1.5)
check("6.6 standardize small", standardize_columns(X), [[-z, -z], [0, 0], [z, z]])

rng = np.random.default_rng(0)
Xr = rng.normal(loc=5, scale=3, size=(50, 4))
check("6.6 column means are 0", lambda: standardize_columns(Xr).mean(axis=0), np.zeros(4))
check("6.6 column stds are 1", lambda: standardize_columns(Xr).std(axis=0), np.ones(4))

### 6d. Vector and matrix products

The dot product $u \cdot v = \sum_i u_i v_i$ is `np.dot(u, v)`, or `u @ v`. For matrices, `A @ B` is
matrix multiplication. Be careful: `A * B` is *elementwise* multiplication, not the matrix product.

The cross product of two 3D vectors is another 3D vector, perpendicular to both:

$$u \times v = (u_1 v_2 - u_2 v_1,\ u_2 v_0 - u_0 v_2,\ u_0 v_1 - u_1 v_0)$$

(written with 0-based indices to match Python). NumPy's version is `np.cross`.

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print(np.dot(u, v), u @ v)
print(np.cross(u, v))

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B =\n", A @ B)
print("A * B =\n", A * B, "   <- elementwise, not matrix multiplication")
print("A.T =\n", A.T)     # transpose

### Exercise 6.7: `dot_loop`
Write the dot product of two 1D arrays with a `for` loop, *without* `np.dot`, `@`, or `np.sum`. Yes,
this is the slow way; that is the point (see 6e). Raise a `ValueError` if the lengths differ.

In [ ]:
def dot_loop(u, v):
    """Return the dot product of u and v, computed with a loop."""
    # YOUR CODE HERE
    pass

In [ ]:
check("6.7 dot_loop small", dot_loop(np.array([1, 2, 3]), np.array([4, 5, 6])), 32)
rng = np.random.default_rng(1)
u, v = rng.normal(size=50), rng.normal(size=50)
check("6.7 dot_loop matches np.dot", dot_loop(u, v), np.dot(u, v))
check_raises("6.7 dot_loop raises on length mismatch",
             lambda: dot_loop(np.ones(3), np.ones(4)), ValueError)

### Exercise 6.8: `cross3`
Write the cross product of two 3D vectors from the formula above, *without* `np.cross`. Return a NumPy
array.

In [ ]:
def cross3(u, v):
    """Return the cross product of 3D vectors u and v."""
    # YOUR CODE HERE
    pass

In [ ]:
i_hat, j_hat, k_hat = np.eye(3)
check("6.8 i x j = k", cross3(i_hat, j_hat), k_hat)
check("6.8 j x i = -k", cross3(j_hat, i_hat), -k_hat)
check("6.8 cross3([1,2,3], [4,5,6])", cross3(np.array([1, 2, 3]), np.array([4, 5, 6])), [-3, 6, -3])
rng = np.random.default_rng(2)
u, v = rng.normal(size=3), rng.normal(size=3)
check("6.8 matches np.cross", cross3(u, v), np.cross(u, v))
check("6.8 perpendicular to u", lambda: np.dot(cross3(u, v), u), 0.0)
check("6.8 returns an array", lambda: isinstance(cross3(u, v), np.ndarray), True)

### Exercise 6.9: `cosine_similarity`
The cosine similarity of two vectors is the cosine of the angle between them:

$$\cos\theta = \frac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}.$$

It is 1 for vectors pointing the same way, 0 for perpendicular vectors, and -1 for opposite vectors.
Use NumPy operations (you may use `np.linalg.norm` here). No loops.

In [ ]:
def cosine_similarity(u, v):
    """Return the cosine of the angle between u and v."""
    # YOUR CODE HERE
    pass

In [ ]:
check("6.9 parallel", cosine_similarity(np.array([1, 2]), np.array([2, 4])), 1.0)
check("6.9 perpendicular", cosine_similarity(np.array([1, 0]), np.array([0, 3])), 0.0)
check("6.9 opposite", cosine_similarity(np.array([1, 1]), np.array([-2, -2])), -1.0)
check("6.9 general", cosine_similarity(np.array([1, 2, 3]), np.array([4, 5, 6])),
      32 / math.sqrt(14 * 77))

### Exercise 6.10: `mat_mul_loops`
This is Exercise 3.10 from Part 1 again, but on NumPy arrays: multiply `A` ($n \times m$) by `B`
($m \times p$) with three nested loops. Start from `np.zeros((n, p))` and use `A.shape` to get the
sizes. Do not use `@`, `np.dot`, or `np.matmul`. Raise a `ValueError` if the inner dimensions do not
match.

In [ ]:
def mat_mul_loops(A, B):
    """Return A @ B computed with explicit loops."""
    # YOUR CODE HERE
    pass

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
check("6.10 mat_mul_loops 2x2", mat_mul_loops(A, B), [[19, 22], [43, 50]])
rng = np.random.default_rng(3)
A, B = rng.normal(size=(3, 4)), rng.normal(size=(4, 2))
check("6.10 matches @ for (3x4)(4x2)", mat_mul_loops(A, B), A @ B)
check("6.10 result shape", lambda: mat_mul_loops(A, B).shape, (3, 2))
check_raises("6.10 raises on bad shapes", lambda: mat_mul_loops(B, A), ValueError)

### Exercise 6.11: `predict`
A linear model with weights `w` and intercept `b` predicts $\hat y = Xw + b$, where each row of `X` is
one sample. Write it in one line with `@`. This is exactly the prediction step of linear regression.

In [ ]:
def predict(X, w, b):
    """Return the linear model predictions X w + b, one per row of X."""
    # YOUR CODE HERE
    pass

In [ ]:
X = np.array([[1, 2], [3, 4], [5, 6]])
w = np.array([0.5, -1.0])
check("6.11 predict", predict(X, w, 2), [0.5, -0.5, -1.5])
check("6.11 predict one feature", predict(np.array([[0], [1], [2]]), np.array([3]), 1), [1, 4, 7])

### Exercise 6.12: `pairwise_distances`
Given `n` points as the rows of `X` (shape `(n, d)`), return the `n` by `n` matrix whose entry `[i, j]`
is the distance between point `i` and point `j`. Do it with broadcasting and **no loops**.

Hint: `X[:, None, :] - X[None, :, :]` has shape `(n, n, d)`, and entry `[i, j]` is the difference vector
between point `i` and point `j`. Square it, sum over the last axis, take the square root.

In [ ]:
def pairwise_distances(X):
    """Return the matrix of Euclidean distances between the rows of X."""
    # YOUR CODE HERE
    pass

In [ ]:
P = np.array([[0, 0], [3, 4], [6, 8]])
check("6.12 pairwise_distances small", pairwise_distances(P), [[0, 5, 10], [5, 0, 5], [10, 5, 0]])

rng = np.random.default_rng(4)
Q = rng.normal(size=(6, 3))
expected = np.array([[np.linalg.norm(Q[i] - Q[j]) for j in range(6)] for i in range(6)])
check("6.12 matches a loop version", pairwise_distances(Q), expected)

### 6e. Why we bother with NumPy

`%timeit` runs a line many times and reports how long it takes. Compare your loop versions with
NumPy's. The gap is usually a factor of 100 or more, and it gets bigger as the arrays grow. This is why
ML code is written with array operations instead of Python loops.

In [ ]:
rng = np.random.default_rng(5)
u, v = rng.normal(size=100_000), rng.normal(size=100_000)
%timeit -n 5 -r 3 dot_loop(u, v)
%timeit -n 5 -r 3 np.dot(u, v)

In [ ]:
A, B = rng.normal(size=(60, 60)), rng.normal(size=(60, 60))
%timeit -n 1 -r 3 mat_mul_loops(A, B)
%timeit -n 5 -r 3 A @ B

## Section 7: Gaussian Elimination

To solve a system of linear equations $Ax = b$, we write the *augmented matrix* $[A \mid b]$ and apply
row operations until it is easy to read off the answer. For example,

$$
\begin{aligned}
 2x + y - z &= 8 \\
-3x - y + 2z &= -11 \\
-2x + y + 2z &= -3
\end{aligned}
\qquad\longrightarrow\qquad
\left[\begin{array}{rrr|r}
 2 &  1 & -1 &   8 \\
-3 & -1 &  2 & -11 \\
-2 &  1 &  2 &  -3
\end{array}\right]
$$

has solution $x = 2,\ y = 3,\ z = -1$.

There are three row operations, and none of them changes the solution:

1. swap two rows,
2. multiply a row by a nonzero number,
3. add a multiple of one row to another row.

**Forward elimination** works column by column. For column $k$, choose a *pivot* row, swap it into
position $k$, and then for every row $i$ below it, subtract $\frac{M_{ik}}{M_{kk}}$ times row $k$ from
row $i$. That makes every entry below the pivot zero. After the last column, the left part of the matrix
is upper triangular.

**Partial pivoting** means the pivot for column $k$ is the row (at or below row $k$) whose entry in
column $k$ has the largest absolute value. This avoids dividing by zero when $M_{kk}$ happens to be 0,
and it also keeps rounding error under control when $M_{kk}$ is tiny. If even the largest available
entry is (essentially) zero, the matrix is singular and there is no unique solution.

**Back substitution** then solves from the bottom up. The last row has only one unknown, so solve for it;
plug that into the row above, which now has only one unknown left; and so on:

$$x_i = \frac{1}{U_{ii}} \Big( c_i - \sum_{j > i} U_{ij} x_j \Big)$$

where $U$ is the triangular part and $c$ is the last column.

In this section, each row operation returns a **new** array and leaves its input alone. (Use
`M.copy()`, or `M.astype(float)`, which also makes a copy, before changing anything.) Work in floats.

### Exercise 7.1: `swap_rows`

In [ ]:
def swap_rows(M, i, j):
    """Return a copy of M with rows i and j swapped."""
    # YOUR CODE HERE
    pass

In [ ]:
M = np.array([[1, 2], [3, 4], [5, 6]])
check("7.1 swap_rows(M, 0, 2)", swap_rows(M, 0, 2), [[5, 6], [3, 4], [1, 2]])
check("7.1 swap a row with itself", swap_rows(M, 1, 1), M)

def _t():
    M = np.array([[1.0, 2.0], [3.0, 4.0]])
    swap_rows(M, 0, 1)
    return M
check("7.1 does not change its input", _t, [[1, 2], [3, 4]])

### Exercise 7.2: `scale_row`

In [ ]:
def scale_row(M, i, c):
    """Return a copy of M with row i multiplied by c."""
    # YOUR CODE HERE
    pass

In [ ]:
M = np.array([[1, 2], [3, 4]])
check("7.2 scale_row(M, 1, 0.5)", scale_row(M, 1, 0.5), [[1, 2], [1.5, 2]])
check("7.2 scale_row(M, 0, -2)", scale_row(M, 0, -2), [[-2, -4], [3, 4]])

def _t():
    M = np.array([[1.0, 2.0], [3.0, 4.0]])
    scale_row(M, 0, 10)
    return M
check("7.2 does not change its input", _t, [[1, 2], [3, 4]])

### Exercise 7.3: `add_multiple`

In [ ]:
def add_multiple(M, src, dst, c):
    """Return a copy of M where row dst has been replaced by (row dst) + c * (row src)."""
    # YOUR CODE HERE
    pass

In [ ]:
M = np.array([[2, 1, 8], [-3, -1, -11]])
check("7.3 add_multiple", add_multiple(M, 0, 1, 1.5), [[2, 1, 8], [0, 0.5, 1]])
check("7.3 source row unchanged", lambda: add_multiple(M, 0, 1, 1.5)[0], [2, 1, 8])

def _t():
    M = np.array([[1.0, 2.0], [3.0, 4.0]])
    add_multiple(M, 0, 1, 5)
    return M
check("7.3 does not change its input", _t, [[1, 2], [3, 4]])

### Exercise 7.4: `augment`
Return the augmented matrix $[A \mid b]$ as a float array with one extra column. `np.column_stack` or
`np.hstack` will help (for `hstack`, `b` must first be reshaped into a column).

In [ ]:
def augment(A, b):
    """Return the n-by-(n+1) float array [A | b]."""
    # YOUR CODE HERE
    pass

In [ ]:
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])
b = np.array([8, -11, -3])
check("7.4 augment", augment(A, b), [[2, 1, -1, 8], [-3, -1, 2, -11], [-2, 1, 2, -3]])
check("7.4 shape", lambda: augment(A, b).shape, (3, 4))
check("7.4 float dtype", lambda: augment(A, b).dtype == np.float64, True)

### Exercise 7.5: `forward_eliminate`
Given an augmented matrix `M` of shape `(n, n+1)`, return a new matrix in upper triangular form (every
entry below the diagonal of the left `n` columns is zero), using partial pivoting as described above.
Use your three row operations. If the best available pivot has absolute value less than `tol`, raise
`ValueError("matrix is singular")`.

Hint: `np.argmax(np.abs(M[k:, k]))` gives the position of the best pivot *counting from row k*.

In [ ]:
def forward_eliminate(M, tol=1e-10):
    """Return an upper triangular matrix row-equivalent to the augmented matrix M."""
    # YOUR CODE HERE
    pass

In [ ]:
M0 = np.array([[2.0, 1, -1, 8], [-3, -1, 2, -11], [-2, 1, 2, -3]])

def below_diagonal(U):
    n = U.shape[0]
    return np.tril(U[:, :n], -1)

check("7.5 result is upper triangular", lambda: below_diagonal(forward_eliminate(M0)), np.zeros((3, 3)))
check("7.5 same shape", lambda: forward_eliminate(M0).shape, (3, 4))
check("7.5 handles a zero in the top-left (needs pivoting)",
      lambda: below_diagonal(forward_eliminate(np.array([[0.0, 1, 2], [1, 1, 3]]))), np.zeros((2, 2)))
check("7.5 pivots on the largest entry",
      lambda: abs(forward_eliminate(np.array([[1.0, 2, 3], [4, 5, 6]]))[0, 0]), 4.0)

def _t():
    M = M0.copy()
    forward_eliminate(M)
    return M
check("7.5 does not change its input", _t, M0)
check_raises("7.5 raises on a singular matrix",
             lambda: forward_eliminate(np.array([[1.0, 2, 3], [2, 4, 6]])), ValueError)
check_raises("7.5 raises on a 3x3 singular matrix",
             lambda: forward_eliminate(augment(np.arange(1.0, 10).reshape(3, 3), np.ones(3))), ValueError)

### Exercise 7.6: `back_substitute`
Given an upper triangular augmented matrix `U` of shape `(n, n+1)`, return the solution `x` as a 1D
array, using the back substitution formula above. Work from the last row up:
`range(n - 1, -1, -1)` counts down from `n - 1` to `0`.

In [ ]:
def back_substitute(U):
    """Solve an upper triangular augmented system and return x."""
    # YOUR CODE HERE
    pass

In [ ]:
check("7.6 back_substitute 2x2", back_substitute(np.array([[2.0, 1, 5], [0, 3, 6]])), [1.5, 2])
check("7.6 back_substitute 1x1", back_substitute(np.array([[4.0, 2]])), [0.5])
U3 = np.array([[1.0, 2, 3, 14],
               [0, 4, 5, 23],
               [0, 0, 6, 18]])
check("7.6 back_substitute 3x3", back_substitute(U3), [1, 2, 3])

### Exercise 7.7: `solve`
Put it all together: augment, eliminate, back-substitute.

In [ ]:
def solve(A, b):
    """Return the solution x of A x = b."""
    # YOUR CODE HERE
    pass

In [ ]:
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])
b = np.array([8, -11, -3])
check("7.7 solve the example", lambda: solve(A, b), [2, 3, -1])
check("7.7 needs pivoting", lambda: solve(np.array([[0, 1], [1, 0]]), np.array([3, 5])), [5, 3])

rng = np.random.default_rng(6)
A5, b5 = rng.normal(size=(5, 5)), rng.normal(size=5)
check("7.7 matches np.linalg.solve (5x5)", lambda: solve(A5, b5), np.linalg.solve(A5, b5))
A20, b20 = rng.normal(size=(20, 20)), rng.normal(size=20)
check("7.7 matches np.linalg.solve (20x20)", lambda: solve(A20, b20), np.linalg.solve(A20, b20))
check("7.7 A x really equals b", lambda: A5 @ solve(A5, b5), b5)
check_raises("7.7 raises on a singular system",
             lambda: solve(np.array([[1, 2], [2, 4]]), np.array([1, 2])), ValueError)

### Exercise 7.8: `fit_line`
Back in Unit 2 we fit lines $y \approx mx + c$ by least squares. The best $(m, c)$ solve the *normal
equations*

$$(X^T X)\, w = X^T y, \qquad X = \begin{bmatrix} x_1 & 1 \\ x_2 & 1 \\ \vdots & \vdots \\ x_n & 1 \end{bmatrix}, \qquad w = \begin{bmatrix} m \\ c \end{bmatrix}.$$

Build $X$ (a column of the $x$ values next to a column of ones), form the 2 by 2 system, and solve it
with **your** `solve`. Return the tuple `(m, c)`.

In [ ]:
def fit_line(x, y):
    """Return (slope, intercept) of the least-squares line through the points (x, y)."""
    # YOUR CODE HERE
    pass

In [ ]:
xs = np.array([0, 1, 2, 3, 4])
check("7.8 exact line y = 2x + 1", lambda: fit_line(xs, 2 * xs + 1), (2, 1))

rng = np.random.default_rng(7)
xn = rng.uniform(0, 10, size=40)
yn = -0.7 * xn + 4 + rng.normal(scale=0.5, size=40)
check("7.8 matches np.polyfit on noisy data", lambda: fit_line(xn, yn), tuple(np.polyfit(xn, yn, 1)))

## Summary

Run the cell below to see how many tests pass. Before submitting, do **Runtime > Restart session and
run all** so the count reflects your final code.

In [ ]:
report()